# Urban Water Model Simulation: Green Roof Scenario

This notebook demonstrates the Green Roof component integration. It runs a baseline simulation (0% Green Roofs) and then allows you to set a percentage of roof area converted to green roofs for comparison.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go

import holoviews as hv
from bokeh.io import output_notebook
hv.extension('bokeh')
output_notebook()

from urbanWater.read_data import read_data
from urbanWater.forcing import read_forcing
from urbanWater.water_model import UrbanWaterModel
from urbanWater.utils import load_config
from urbanWater.scenario_manager import Scenario, run_scenario
from urbanWater.postprocess import extract_local_results

from urbanWater.viz import (
    plot_aggregated_results,
    create_map_base,
    create_dynamic_map,
    create_flows,
    create_reuse_flows
)

## Configuration & Baseline Run (0% Green Roof)

Load configuration and run the model with 0% green roof coverage to establish a baseline.

In [ ]:
config = load_config('.', 'default', 'config.yaml')

print(f"Experiment: Green Roof Simulation")
print(f"Simulation period: {config.simulation.start_date} - {config.simulation.end_date}")

geo_file = Path(config.input_directory) / Path(config.files.geo)
background_file = Path(config.geodata_directory) / config.files.background_shapefile

# Read Forcing
forcing_data = read_forcing(config)

# --- Baseline Run (0%) ---
print("Running Baseline Simulation (0% Green Roof)...")
config.model.calibration['greenroof_coverage'] = 0.0
model_params_0, reuse_settings, demand_data, soil_data, et_data, flow_paths = read_data(config)

scenario_0 = Scenario(name='baseline_0', description='0% Green Roof Coverage')
model_data_0 = {
    'flow_paths': flow_paths, 
    'soil_data': soil_data,
    'et_data': et_data,
    'demand_data': demand_data,
    'reuse_settings': reuse_settings,
    'direction': config.grid.direction
}

_, results_0 = run_scenario((scenario_0.name, model_params_0, forcing_data, model_data_0, None, None, True))
print("Baseline Simulation Complete.")

# Show Base Map
base_map = create_map_base(geo_file, background_file, flow_paths)
base_map.show()

## Green Roof Scenario

Select the Green Roof Coverage percentage (must be > 0% to see a difference).

In [ ]:
coverage_slider = widgets.FloatSlider(
    value=50.0,
    min=0.0,
    max=100.0,
    step=5.0,
    description='Green Roof %:',
    continuous_update=False
)
display(coverage_slider)

## Run Green Roof Simulation

Execute the simulation using the selected green roof coverage.

In [ ]:
coverage = coverage_slider.value
print(f"Running simulation with {coverage}% Green Roof coverage...")

# Update configuration
config.model.calibration['greenroof_coverage'] = coverage

# Reload parameters with new coverage
model_params, reuse_settings, demand_data, soil_data, et_data, flow_paths = read_data(config)

# Create scenario 
scenario_gr = Scenario(
    name=f'greenroof_{int(coverage)}',
    description=f'{coverage}% Green Roof Coverage'
)

# Create model data dict
model_data = {
    'flow_paths': flow_paths, 
    'soil_data': soil_data,
    'et_data': et_data,
    'demand_data': demand_data,
    'reuse_settings': reuse_settings,
    'direction': config.grid.direction
}

# Run scenario
_, results_gr = run_scenario((scenario_gr.name, model_params, forcing_data, model_data, None, None, True))

print("Green Roof Simulation Complete.")

## Comparison: Baseline vs. Green Roof

Compare key hydrological indicators between the Baseline (0%) and your selected Green Roof scenario.

In [ ]:
def plot_comparison(res_base, res_scenario, scenario_label):
    agg_base = res_base['aggregated']
    agg_scen = res_scenario['aggregated']
    
    fig = go.Figure()
    
    # 1. Stormwater Runoff Comparison
    fig.add_trace(go.Scatter(
        x=agg_base.index, 
        y=agg_base['stormwater'].pint.magnitude,
        name='Runoff (Baseline)',
        line=dict(color='blue', dash='solid')
    ))
    fig.add_trace(go.Scatter(
        x=agg_scen.index, 
        y=agg_scen['stormwater'].pint.magnitude,
        name=f'Runoff ({scenario_label})',
        line=dict(color='darkviolet', dash='dot')
    ))
    
    # 2. Evapotranspiration Comparison
    et_base = (agg_base['evaporation'] + agg_base['transpiration']).pint.magnitude
    et_scen = (agg_scen['evaporation'] + agg_scen['transpiration']).pint.magnitude
    
    fig.add_trace(go.Scatter(
        x=agg_base.index, 
        y=et_base,
        name='ET (Baseline)',
        line=dict(color='green', dash='solid'),
        visible='legendonly'
    ))
    fig.add_trace(go.Scatter(
        x=agg_scen.index, 
        y=et_scen,
        name=f'ET ({scenario_label})',
        line=dict(color='darkorange', dash='dot'),
        visible='legendonly'
    ))
    
    fig.update_layout(
        title=f"Comparison: Baseline vs. {scenario_label}",
        xaxis_title="Date",
        yaxis_title="Flow [m³/day]",
        height=500,
        hovermode='x unified'
    )
    return fig

comp_fig = plot_comparison(results_0, results_gr, f"{coverage}% Green Roof")
comp_fig.show()

# Summary Table
# Note: .sum() returns a scalar Quantity, so we use .magnitude directly instead of .pint.magnitude
summary = pd.DataFrame({
    'Baseline (0%)': {
        'Total Runoff [m3]': results_0['aggregated']['stormwater'].sum().magnitude,
        'Total ET [m3]': (results_0['aggregated']['evaporation'] + results_0['aggregated']['transpiration']).sum().magnitude
    },
    f'Green Roof ({coverage}%)': {
        'Total Runoff [m3]': results_gr['aggregated']['stormwater'].sum().magnitude,
        'Total ET [m3]': (results_gr['aggregated']['evaporation'] + results_gr['aggregated']['transpiration']).sum().magnitude
    }
})
summary.loc['Difference (%)'] = (summary.iloc[1] - summary.iloc[0]) / summary.iloc[0] * 100
display(summary.round(2))

## Visualize Results (Green Roof Scenario)

Detailed visualizations for the selected Green Roof scenario.

In [ ]:
fig = plot_aggregated_results(results_gr['aggregated'], results_gr['forcing'])
fig.show()

## Generate Interactive Maps

Create interactive map visualizations of the results.

In [ ]:
area_variables = [
    'evapotranspiration',
    'imported_water',
    'baseflow',
    'deep_seepage',
    'stormwater_runoff',
    'sewerage_discharge',
    'groundwater',
    'vadose_moisture'
]

gdf_geometry = gpd.read_file(geo_file)
local_results = extract_local_results(results_gr)
time_series_data = local_results[area_variables].unstack(level='cell')
time_series_data.index = pd.to_datetime(time_series_data.index)

dynamic_map = create_dynamic_map(gdf_geometry, background_file, area_variables, time_series_data)
dynamic_map.show()

## Generate Alluvial diagrams

In [ ]:
sankey_fig = create_flows(results_gr, flow_paths, viz_type='sankey')
sankey_fig.show()

In [ ]:
sankey_fig = create_reuse_flows(results_gr, viz_type='sankey')
sankey_fig.show()

## Generate Chord diagrams

In [ ]:
chord_fig = create_flows(results_gr, flow_paths, viz_type='chord') 
chord_fig

In [ ]:
chord_fig = create_reuse_flows(results_gr, viz_type='chord') 
chord_fig